# Sherm Quanty — Two-Tier Regime Engines: Test & Backtest

**Self-contained notebook** for Google Colab / Kaggle. Nothing to upload — both regime engines are embedded inline:

| Engine | Timeframe | Output | Origin |
|--------|-----------|--------|--------|
| **1. Macro** | Daily | `BULL` / `NON_BULL` | Cloned from VortexMomentum V14 (rule-based: MA200 + HH/HL swing + ADX proxy) |
| **2. Tactical** | 2-hour | `H_BULL / L_BULL / SIDEWAYS / L_BEAR / H_BEAR` | Our 5-state Gaussian HMM |

Design principle: the **daily macro** engine is the slow, stable climate filter; the **2h tactical** HMM is the fast, actionable swing regime. This notebook builds both, visualises both, backtests both, and compares their detection timing.

Both engines fall back to calibrated synthetic data if yfinance is blocked (shared Colab/Kaggle IPs are often rate-limited), so the notebook always runs end-to-end.

In [ ]:
%pip install -q hmmlearn yfinance

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)
np.random.seed(42)

DATA_START = '2010-01-01'
REGIME_LABELS = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']
REGIME_COLORS = {
    'H_BULL': '#006400', 'L_BULL': '#90EE90', 'SIDEWAYS': '#808080',
    'L_BEAR': '#FFB6C1', 'H_BEAR': '#8B0000',
    'BULL': '#2E8B57', 'NON_BULL': '#B0B0B0',
}
print('imports ok')

---
# ENGINE 1 — Daily Macro (VortexMomentum V14 clone)

Rule-based, no ML. `BULL` only when **all three** hold on 3-business-day resampled close:
1. price > MA200 (67 three-day bars ≈ 200 daily bars)
2. HH/HL swing structure over the last 20 bars
3. ADX proxy > 0.10 (directional strength)

`confidence` = fraction of the three conditions met (a graded extension over the original binary detector).

In [ ]:
# --- macro config (VortexMomentum V14 — do not tune without a backtest) ---
SWING_LOOKBACK      = 20
ADX_PERIOD          = 14
MA200_BARS_3D       = 67
ADX_PROXY_THRESHOLD = 0.10
MIN_3D_BARS         = 75


def macro_generate_synthetic_daily(start=DATA_START):
    """Regime-blocked GBM daily close, calibrated to documented NSE history."""
    rng = np.random.default_rng(42)
    dates = pd.bdate_range(start=start, end=pd.Timestamp.today().normalize())
    REGIMES = [
        ('2010-01-01', '2010-10-31',  0.0008, 0.0095), ('2010-11-01', '2011-12-31', -0.0003, 0.0135),
        ('2012-01-01', '2013-08-31',  0.0002, 0.0085), ('2013-09-01', '2014-04-30',  0.0003, 0.0090),
        ('2014-05-01', '2015-12-31',  0.0010, 0.0075), ('2016-01-01', '2016-02-28', -0.0007, 0.0120),
        ('2016-03-01', '2016-10-31',  0.0008, 0.0070), ('2016-11-01', '2016-12-31', -0.0008, 0.0130),
        ('2017-01-01', '2017-12-31',  0.0009, 0.0065), ('2018-01-01', '2019-03-31', -0.0002, 0.0100),
        ('2019-04-01', '2020-01-31',  0.0005, 0.0085), ('2020-02-01', '2020-03-23', -0.0055, 0.0250),
        ('2020-03-24', '2020-12-31',  0.0015, 0.0130), ('2021-01-01', '2021-10-31',  0.0010, 0.0072),
        ('2021-11-01', '2022-06-30', -0.0003, 0.0105), ('2022-07-01', '2023-12-31',  0.0007, 0.0082),
        ('2024-01-01', '2024-05-31',  0.0008, 0.0075), ('2024-06-01', '2026-12-31',  0.0005, 0.0090),
    ]
    rmap = {}
    for s, e, mu, sig in REGIMES:
        s, e = pd.Timestamp(s), pd.Timestamp(e)
        for d in dates:
            if s <= d <= e:
                rmap[d] = (mu, sig)
    level, vals = 5200.0, []
    for d in dates:
        mu, sig = rmap.get(d, (0.0004, 0.0090))
        level *= np.exp(rng.normal(mu, sig)); vals.append(level)
    return pd.Series(vals, index=dates, name='nifty'), True


def macro_load_daily(start=DATA_START):
    """Daily Nifty close from yfinance (multiple India-index fallbacks) or synthetic."""
    try:
        import yfinance as yf
        for tk in ['^NSEI', '^CRSLDX', 'NIFTYBEES.NS']:
            raw = yf.download(tk, start=start, auto_adjust=True, progress=False)
            if len(raw) > 500:
                s = raw['Close'].squeeze().dropna()
                s.index = pd.to_datetime(s.index).normalize(); s.name = 'nifty'
                print(f'macro: yfinance {tk} -> {len(s)} daily bars')
                return s, False
        raise ValueError('no ticker returned enough data')
    except Exception as e:
        print(f'macro: yfinance unavailable ({e}); synthetic daily data.')
        return macro_generate_synthetic_daily(start)

In [ ]:
def macro_detect_on_date(nifty_close, date):
    """VortexMomentum V14 detector -> ('BULL'|'NON_BULL', confidence in {0,.33,.67,1})."""
    try:
        data_3d = nifty_close.loc[:date].resample('3B').last().dropna()
        if len(data_3d) < MIN_3D_BARS:
            return 'NON_BULL', 0.0
        c = data_3d.values
        above_ma200 = c[-1] > c[-MA200_BARS_3D:].mean()
        w, mid = c[-SWING_LOOKBACK:], SWING_LOOKBACK // 2
        hh_hl = (w[mid:].max() > w[:mid].max()) and (w[mid:].min() > w[:mid].min())
        atr = np.abs(np.diff(c[-(ADX_PERIOD + 2):])).mean()
        net = abs(c[-1] - c[-(ADX_PERIOD + 1)])
        trending = (net / (atr * ADX_PERIOD) if atr > 0 else 0.0) > ADX_PROXY_THRESHOLD
        n = sum(bool(x) for x in (above_ma200, hh_hl, trending))
        return ('BULL' if n == 3 else 'NON_BULL'), round(n / 3.0, 4)
    except Exception:
        return 'NON_BULL', 0.0


def macro_classify(nifty_close):
    labels, confs = [], []
    for d in nifty_close.index:
        lbl, cf = macro_detect_on_date(nifty_close, d)
        labels.append(lbl); confs.append(cf)
    df = pd.DataFrame({'daily_regime_state': labels, 'daily_regime_confidence': confs,
                       'nifty_close': nifty_close.values}, index=nifty_close.index)
    df.index.name = 'date'
    flipped = df['daily_regime_state'] != df['daily_regime_state'].shift(1)
    flipped.iloc[0] = False
    df['daily_transition_warning_flag'] = flipped.values.astype(bool)
    return df


nifty_daily, MACRO_SYNTH = macro_load_daily()
macro_df = macro_classify(nifty_daily)
print('\nrows:', len(macro_df), '| synthetic:', MACRO_SYNTH)
print('\ndaily macro distribution:')
print(macro_df['daily_regime_state'].value_counts().to_string())
print('BULL fraction: {:.1%} | transitions: {}'.format(
    (macro_df['daily_regime_state'] == 'BULL').mean(),
    int(macro_df['daily_transition_warning_flag'].sum())))
macro_df.tail()

In [ ]:
def plot_binary_regime(df, state_col, price_col, title, synthetic=False):
    fig, ax = plt.subplots(figsize=(20, 7))
    fig.suptitle(title + ('  [SYNTHETIC DATA]' if synthetic else ''), fontsize=13)
    regimes, idx, start_i = df[state_col].values, df.index, 0
    for i in range(1, len(regimes)):
        if regimes[i] != regimes[i - 1]:
            ax.axvspan(idx[start_i], idx[i - 1], alpha=0.35,
                       color=REGIME_COLORS.get(regimes[start_i], '#808080'), lw=0)
            start_i = i
    ax.axvspan(idx[start_i], idx[-1], alpha=0.35,
               color=REGIME_COLORS.get(regimes[start_i], '#808080'), lw=0)
    ax.plot(df.index, df[price_col], color='black', lw=0.8)
    ax.set_ylabel('Nifty 50')
    ax.legend(handles=[mpatches.Patch(color=REGIME_COLORS['BULL'], alpha=0.7, label='BULL'),
                       mpatches.Patch(color=REGIME_COLORS['NON_BULL'], alpha=0.7, label='NON_BULL')],
              loc='upper left')
    plt.tight_layout(); plt.show()

plot_binary_regime(macro_df, 'daily_regime_state', 'nifty_close',
                   'Engine 1 — Daily Macro (VortexMomentum V14)', synthetic=MACRO_SYNTH)

---
# ENGINE 2 — 2-Hour Tactical (5-state Gaussian HMM)

Nine swing features → `StandardScaler` fit on the training window only → 5-state full-covariance HMM → states labelled by a **composite bullishness score** (not mean-return alone) so early momentum/vol expansion is caught sooner. `SIDEWAYS` override when the leading-state probability < 0.50; transition warnings on confidence drops or VIX spikes.

yfinance has no native 2h interval, so we pull **60-minute** bars (capped ~730 days) and resample to 2h — same as the repo engine. Synthetic 2h fallback if offline.

In [ ]:
# --- tactical config ---
TRAIN_FRACTION          = 0.70
N_STATES                = 5
CONFIDENCE_THRESHOLD_L  = 0.50
HMM_PROB_DROP_THRESHOLD = 0.20
VIX_SPIKE_THRESHOLD     = 0.25
BARS_PER_DAY            = 3
MOM_1D, MOM_3D, MOM_5D  = 1 * BARS_PER_DAY, 3 * BARS_PER_DAY, 5 * BARS_PER_DAY
VOL_WIN, VOL_FAST, VOL_SLOW = 10, 5, 20
SWING_WIN               = 20
FEATURE_COLS = ['ret_2h', 'mom_1d', 'mom_3d', 'mom_5d',
                'vol_2h', 'vol_expansion', 'vix_chg', 'drawdown', 'dist_ma']


def tac_generate_synthetic_2h():
    """Regime-blocked GBM + OU VIX at 2h cadence (~3 bars/business day, ~730 days)."""
    rng = np.random.default_rng(42)
    days = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=520)
    idx = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in days for h in (10, 12, 14)])
    REGIMES = [  # frac_start, frac_end, mu, sigma, vix_mu, vix_vf
        (0.00, 0.15,  0.0004, 0.0035, 14.0, 0.6), (0.15, 0.30, -0.0006, 0.0060, 22.0, 1.1),
        (0.30, 0.45,  0.0001, 0.0030, 16.0, 0.7), (0.45, 0.55, -0.0030, 0.0110, 45.0, 2.5),
        (0.55, 0.75,  0.0006, 0.0050, 20.0, 1.0), (0.75, 0.90,  0.0005, 0.0032, 13.5, 0.6),
        (0.90, 1.00, -0.0002, 0.0045, 18.0, 0.9),
    ]
    n, level, prev_vix = len(idx), 18000.0, 15.0
    nvals, vvals = [], []
    for i in range(n):
        frac = i / n
        mu, sig, vm, vf = 0.0002, 0.0040, 16.0, 0.8
        for fs, fe, m, s, v, f in REGIMES:
            if fs <= frac < fe:
                mu, sig, vm, vf = m, s, v, f; break
        level *= np.exp(rng.normal(mu, sig))
        prev_vix = max(prev_vix + 0.05 * (vm - prev_vix) + rng.normal(0, vm * 0.08 * vf), 8.0)
        nvals.append(level); vvals.append(prev_vix)
    return (pd.Series(nvals, index=idx, name='nifty'),
            pd.Series(vvals, index=idx, name='vix'), True)


def _yf_hourly_to_2h(ticker):
    import yfinance as yf
    raw = yf.download(ticker, interval='60m', period='730d', auto_adjust=True, progress=False)
    if raw is None or len(raw) == 0:
        return None
    close = raw['Close'].squeeze().dropna()
    close.index = pd.to_datetime(close.index)
    return close.resample('2h').last().dropna()


def tac_load_2h():
    """2h Nifty + VIX from yfinance (60m->2h) or synthetic."""
    try:
        nifty = _yf_hourly_to_2h('^NSEI')
        if nifty is not None and len(nifty) > 200:
            vix = _yf_hourly_to_2h('^INDIAVIX')
            vix = (vix.reindex(nifty.index).ffill().bfill() if vix is not None and len(vix)
                   else pd.Series(15.0, index=nifty.index))
            nifty.name, vix.name = 'nifty', 'vix'
            print(f'tactical: yfinance 60m->2h -> {len(nifty)} bars')
            return nifty, vix, False
        raise ValueError('too few 2h bars')
    except Exception as e:
        print(f'tactical: yfinance unavailable ({e}); synthetic 2h data.')
        return tac_generate_synthetic_2h()

In [ ]:
def tac_build_features(nifty, vix, train_fraction=TRAIN_FRACTION):
    log_ret = np.log(nifty / nifty.shift(1))
    df = pd.DataFrame(index=nifty.index)
    df['ret_2h']        = log_ret
    df['mom_1d']        = log_ret.rolling(MOM_1D).sum()
    df['mom_3d']        = log_ret.rolling(MOM_3D).sum()
    df['mom_5d']        = log_ret.rolling(MOM_5D).sum()
    df['vol_2h']        = log_ret.rolling(VOL_WIN).std()
    df['vol_expansion'] = log_ret.rolling(VOL_FAST).std() / log_ret.rolling(VOL_SLOW).std()
    df['vix_chg']       = (vix - vix.shift(1)) / vix.shift(1)
    df['drawdown']      = (nifty.rolling(SWING_WIN).max() - nifty) / nifty.rolling(SWING_WIN).max()
    df['dist_ma']       = (nifty - nifty.rolling(SWING_WIN).mean()) / nifty.rolling(SWING_WIN).mean()
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    X_raw = df[FEATURE_COLS].values
    n_train = max(int(len(X_raw) * train_fraction), 50)
    scaler = StandardScaler().fit(X_raw[:n_train])
    return scaler.transform(X_raw), df.index, scaler, df, n_train


def tac_composite_scores(model):
    m = model.means_
    ret, mom1, mom3, mom5, vol, volx, vixc, dd, distma = [m[:, i] for i in range(9)]
    return ret + 0.4 * (mom1 + mom3 + mom5) - vol - volx - vixc - dd + distma


def tac_fit_and_classify(X, dates, nifty, vix, n_train):
    model = hmm.GaussianHMM(n_components=N_STATES, covariance_type='full', n_iter=2000,
                            random_state=42, init_params='stmc', params='stmc')
    model.fit(X[:n_train])
    print('tactical HMM converged:', model.monitor_.converged)
    order = np.argsort(tac_composite_scores(model))
    state_labels = {order[0]: 'H_BEAR', order[1]: 'L_BEAR', order[2]: 'SIDEWAYS',
                    order[3]: 'L_BULL', order[4]: 'H_BULL'}
    states, probs = model.predict(X), model.predict_proba(X)
    conf, lead = probs.max(axis=1), probs.argmax(axis=1)
    regime = np.array(['SIDEWAYS' if conf[i] < CONFIDENCE_THRESHOLD_L else state_labels[lead[i]]
                       for i in range(len(dates))])
    prob_cols = {lab: np.zeros(len(dates)) for lab in REGIME_LABELS}
    for raw, lab in state_labels.items():
        prob_cols[lab] = probs[:, raw]
    conf_s, vix_al = pd.Series(conf, index=dates), vix.reindex(dates)
    twf = ((conf_s.shift(1) - conf_s) > HMM_PROB_DROP_THRESHOLD) | \
          (((vix_al - vix_al.shift(1)) / vix_al.shift(1)) > VIX_SPIKE_THRESHOLD)
    twf.iloc[0] = False
    out = pd.DataFrame({
        'tactical_regime_state': regime, 'tactical_regime_confidence': conf,
        'tactical_transition_warning_flag': twf.values.astype(bool),
        'hmm_state_int': states.astype(int), 'nifty_close': nifty.reindex(dates).values,
        'vix_close': vix_al.values,
        **{f'prob_{lab}': prob_cols[lab] for lab in REGIME_LABELS},
    }, index=dates)
    out.index.name = 'date'
    return out, model, state_labels


nifty_2h, vix_2h, TAC_SYNTH = tac_load_2h()
Xt, dates_t, scaler_t, feat_t, n_train_t = tac_build_features(nifty_2h, vix_2h)
print(f'feature matrix: {Xt.shape} | train: {n_train_t} | oos: {len(Xt) - n_train_t}')
tac_df, tac_model, tac_state_labels = tac_fit_and_classify(Xt, dates_t, nifty_2h, vix_2h, n_train_t)
tac_split = dates_t[n_train_t]

print('\ntactical (2h) distribution:')
print(tac_df['tactical_regime_state'].value_counts().reindex(REGIME_LABELS).to_string())
print('transition warnings:', int(tac_df['tactical_transition_warning_flag'].sum()))
tac_df.tail()

In [ ]:
def plot_tactical(df, split_date=None, synthetic=False):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 9),
                                   gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    fig.suptitle('Engine 2 — 2h Tactical (5-state HMM)' + ('  [SYNTHETIC DATA]' if synthetic else ''),
                 fontsize=13)
    regimes, idx, start_i = df['tactical_regime_state'].values, df.index, 0
    for i in range(1, len(regimes)):
        if regimes[i] != regimes[i - 1]:
            c = REGIME_COLORS.get(regimes[start_i], '#808080')
            ax1.axvspan(idx[start_i], idx[i - 1], alpha=0.35, color=c, lw=0)
            ax2.axvspan(idx[start_i], idx[i - 1], alpha=0.20, color=c, lw=0)
            start_i = i
    c = REGIME_COLORS.get(regimes[start_i], '#808080')
    ax1.axvspan(idx[start_i], idx[-1], alpha=0.35, color=c, lw=0)
    ax2.axvspan(idx[start_i], idx[-1], alpha=0.20, color=c, lw=0)
    ax1.plot(df.index, df['nifty_close'], color='black', lw=0.8); ax1.set_ylabel('Nifty (2h)')
    ax2.plot(df.index, df['tactical_regime_confidence'], color='navy', lw=0.8)
    ax2.axhline(CONFIDENCE_THRESHOLD_L, color='orange', ls='--', lw=0.7)
    ax2.set_ylabel('confidence'); ax2.set_ylim(0, 1.05)
    for d in df.index[df['tactical_transition_warning_flag']]:
        ax1.axvline(d, color='purple', ls='--', lw=0.35, alpha=0.35)
    if split_date is not None:
        for ax in (ax1, ax2): ax.axvline(split_date, color='blue', lw=1.2)
        ax1.text(split_date, ax1.get_ylim()[1], '  train | test', color='blue', va='top', fontsize=9)
    ax1.legend(handles=[mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r) for r in REGIME_LABELS],
               loc='upper left', ncol=5, fontsize=8)
    plt.tight_layout(); plt.show()

plot_tactical(tac_df, split_date=tac_split, synthetic=TAC_SYNTH)

---
# Backtest both engines

Same simple long/flat rule for a like-for-like comparison, positions taken on the **previous** bar's regime (no look-ahead):
- **Macro**: long when `BULL`, else flat (daily bars)
- **Tactical**: long when `H_BULL` or `L_BULL`, else flat (2h bars)

vs buy-and-hold on the same series. Sharpe is annualised with the engine's own bar frequency (252 daily / ~756 two-hourly bars per year).

In [ ]:
def run_backtest(df, state_col, long_states, price_col='nifty_close', ppy=252):
    d = df.copy()
    d['mkt_ret'] = d[price_col].pct_change().fillna(0.0)
    d['position'] = d[state_col].isin(long_states).shift(1).fillna(False).astype(int)
    d['strat_ret'] = d['position'] * d['mkt_ret']
    return d

def stat_block(returns, ppy):
    eq = (1 + returns).cumprod()
    years = len(returns) / ppy
    return dict(
        total_return=eq.iloc[-1] - 1,
        cagr=eq.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan,
        sharpe=returns.mean() / returns.std() * np.sqrt(ppy) if returns.std() > 0 else np.nan,
        max_drawdown=(eq / eq.cummax() - 1).min())

def report(name, d, ppy):
    s, b = stat_block(d['strat_ret'], ppy), stat_block(d['mkt_ret'], ppy)
    print(f'\n=== {name} ({d.index[0].date()} -> {d.index[-1].date()}, {len(d)} bars) ===')
    print(f"{'metric':<16}{'regime long/flat':>18}{'buy & hold':>16}")
    for k in ('total_return', 'cagr', 'sharpe', 'max_drawdown'):
        f = (lambda v: f'{v:>17.2f}') if k == 'sharpe' else (lambda v: f'{v:>17.2%}')
        print(f'{k:<16}{f(s[k])} {f(b[k])}')
    print(f"{'time in market':<16}{d['position'].mean():>17.1%} {1.0:>17.1%}")

# --- Engine 1: macro daily ---
macro_bt = run_backtest(macro_df, 'daily_regime_state', {'BULL'}, ppy=252)
print('#' * 30, 'ENGINE 1 — MACRO (daily)', '#' * 30)
report('MACRO full sample', macro_bt, 252)

# --- Engine 2: tactical 2h ---
tac_bt = run_backtest(tac_df, 'tactical_regime_state', {'H_BULL', 'L_BULL'}, ppy=252 * BARS_PER_DAY)
print('\n' + '#' * 30, 'ENGINE 2 — TACTICAL (2h)', '#' * 30)
report('TACTICAL full sample', tac_bt, 252 * BARS_PER_DAY)
report('TACTICAL out-of-sample', tac_bt.loc[tac_split:], 252 * BARS_PER_DAY)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(22, 7))
for ax, d, title, split in [(a1, macro_bt, 'Engine 1 — Macro (daily)', None),
                            (a2, tac_bt, 'Engine 2 — Tactical (2h)', tac_split)]:
    ax.plot(d.index, (1 + d['mkt_ret']).cumprod(),  label='Buy & Hold', color='gray', lw=1.2)
    ax.plot(d.index, (1 + d['strat_ret']).cumprod(), label='Regime long/flat', color='darkgreen', lw=1.4)
    if split is not None:
        ax.axvline(split, color='blue', lw=1.1); ax.text(split, ax.get_ylim()[1], '  test', color='blue', va='top')
    ax.set_yscale('log'); ax.set_title(title); ax.legend(loc='upper left')
fig.suptitle('Equity curves — growth of 1 (log scale)' +
             ('  [SYNTHETIC DATA]' if (MACRO_SYNTH or TAC_SYNTH) else ''), fontsize=13)
plt.tight_layout(); plt.show()

---
# Compare: does the tactical engine turn earlier?

The design goal is that the fast **2h tactical** engine flags high-opportunity phases *before* the slow **daily macro** engine. This table finds the first date each engine enters the target regime inside documented high-opportunity windows. `lag(d) > 0` means tactical was earlier.

> On synthetic data the two series are independent draws, so the comparison is only meaningful once you run with real yfinance data (`MACRO_SYNTH == False` and `TAC_SYNTH == False`).

In [ ]:
BENCHMARK_EVENTS = [
    ('2020 COVID crash',    '2020-02-15', '2020-03-23', 'H_BEAR'),
    ('2020 recovery',       '2020-03-24', '2020-12-31', 'H_BULL'),
    ('2021 bull',           '2021-01-01', '2021-10-31', 'H_BULL'),
    ('2022 bear/rate-hike', '2021-11-01', '2022-06-30', 'H_BEAR'),
    ('2024 rally',          '2024-01-01', '2024-05-31', 'H_BULL'),
]

def first_hit(df, col, target, start, end):
    w = df.loc[(df.index >= pd.Timestamp(start)) & (df.index <= pd.Timestamp(end))]
    hits = w.index[w[col] == target]
    return hits[0] if len(hits) else None

print(f"{'Event':<22}{'target':<9}{'macro 1st':<13}{'tactical 1st':<15}{'lag(d)':<7}")
print('-' * 66)
for name, start, end, target in BENCHMARK_EVENTS:
    macro_target = 'BULL' if target == 'H_BULL' else 'NON_BULL'
    d1 = first_hit(macro_df, 'daily_regime_state', macro_target, start, end)
    t1 = first_hit(tac_df, 'tactical_regime_state', target, start, end)
    lag = f'{(d1 - t1).days:+d}' if (d1 is not None and t1 is not None) else ''
    print(f"{name:<22}{target:<9}"
          f"{str(d1.date()) if d1 is not None else '—':<13}"
          f"{str(t1.date()) if t1 is not None else '—':<15}{lag:<7}")
print('-' * 66)
print('lag(d) > 0 => tactical (2h) detected the phase earlier than daily macro.')
if MACRO_SYNTH or TAC_SYNTH:
    print('\nNOTE: at least one engine is on SYNTHETIC data — comparison is illustrative only.')

In [ ]:
# Save both classified histories for download
macro_df.to_csv('macro_regime_history.csv')
tac_df.to_csv('tactical_regime_history.csv')
print('saved -> macro_regime_history.csv, tactical_regime_history.csv')
print('\nlatest macro   :', macro_df.iloc[-1]['daily_regime_state'],
      f"({macro_df.iloc[-1]['daily_regime_confidence']:.0%})")
print('latest tactical:', tac_df.iloc[-1]['tactical_regime_state'],
      f"({tac_df.iloc[-1]['tactical_regime_confidence']:.0%})")

---
**Notes**
- If either engine printed a *synthetic* fallback, yfinance was blocked on this IP — the numbers are illustrative. Re-run later or supply your own CSVs for real validation.
- The macro engine here matches `regime_engine_macro.py`; the tactical engine matches `regime_engine_tactical.py` (same 9 features, composite labelling, SIDEWAYS override, transition-warning logic). The only difference vs production is that this notebook keeps everything in memory instead of writing parquet/model files.
- Real-data sanity checks: 2014/2021 skew bullish, Mar-2020 a sharp bear, 2022 bearish; the tactical engine should generally turn a few days before the daily macro flip.